# Part e: Classification on MNIST


In [6]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import numpy as np
from src.network import NeuralNetwork
from src.functions import softmax, ReLU, ReLU_der, identity_der, cross_entropy, softmax_cross_entropy_der
from src.optimizers import Adam
from src.data import load_mnist, make_mnist_split, one_hot


In [ ]:
x_train, y_train, x_test, y_test = load_mnist("../data/mnist.npz") # loading the mnist dataset, flattening the image pixel to shape (784,)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape) # checking the shapes of training set, test set and true digits

(60000, 784) (60000,) (10000, 784) (10000,)


In [ ]:
# splitting the training set of 6000 images into 5000 training images, 1000 validation images and 1000 test images
x_tr, x_val, x_test, y_tr, y_val, y_test = make_mnist_split("../data/mnist.npz") 
Y_tr = one_hot(y_tr) # converts digit to one hot row (10,) with 1 in the true digit position
print(x_tr.shape, x_val.shape, x_test.shape, Y_tr.shape)

(50000, 784) (10000, 784) (10000, 784) (50000, 10)


In [10]:
# test of the functions in src/metrics.py on a tiny example
import numpy as np
from sklearn import metrics as skm
from src.metrics import accuracy, confusion_matrix_multiclass, per_class_from_confusion, standard_error

y_true = np.array([3, 7, 1, 7])
y_pred = np.array([3, 2, 1, 7])

print(accuracy(y_true, y_pred), skm.accuracy_score(y_true, y_pred))          # both 0.75

C = confusion_matrix_multiclass(y_true, y_pred, 10)
print(np.array_equal(C, skm.confusion_matrix(y_true, y_pred, labels=range(10))))   # True

precision, recall = per_class_from_confusion(C)
print(np.allclose(precision, skm.precision_score(y_true, y_pred, labels=range(10), average=None, zero_division=0)))  # True
print(np.allclose(recall, skm.recall_score(y_true, y_pred, labels=range(10), average=None, zero_division=0)))        # True

print(standard_error(0.75, 4))    # about 0.2165

0.75 0.75
True
True
True
0.21650635094610965


In [ ]:
# Logistic regression network, 784 -> 10 (no hidden layers), softmax output + cross entropy
'''
Functions used:

softmax turns the 10 outputs into probabilities that sum to 1.
cross_entropy is the cost function for classification.
softmax_cross_entropy_der is the derivative used in backpropagation for the last layer. 
identity_der is a derivative that is always 1
The Adam optimizer updates the weights during training.
'''
# Creating the network
logreg = NeuralNetwork(784, [10], [softmax], [identity_der],      # identity_der: the softmax derivative is already in softmax_cross_entropy_der
                       cross_entropy, softmax_cross_entropy_der, Adam(gamma=0.001))

# Training the network with stochastic gradient descent, gives the training cost after each epoch
history = logreg.train(x_tr, Y_tr, epochs=5, batch_size=100, rng=np.random.default_rng(2026))


# Finding the predicted numbers for each image in the validation dataset
# logreg.predict(x_val) gives 10 probabilities for each of the 10 000 validation images, with shape (10000, 10).
# np.argmax(..., axis=1) picks, for each image, the position of the largest probability, i.e. the predicted digit. The result has shape (10000,).
pred_val = np.argmax(logreg.predict(x_val), axis=1)    # predicted digit = the output with the highest probability

# compares the predicted digits with the true digits y_val, and prints the fraction that are correct
print("validation accuracy for logistic regression:", accuracy(y_val, pred_val))

validation accuracy for logistic regression: 0.9154


In [ ]:
# Network with one hidden layer of 100 ReLU nodes: 784 -> 100 -> 10
# same principle as for logistic regression, but adding activation function ReLU for the hidden layer and outputs 100 and 10

net_layer = NeuralNetwork(784, [100, 10], [ReLU, softmax], [ReLU_der, identity_der],
                    cross_entropy, softmax_cross_entropy_der, Adam(gamma=0.001))   # a new Adam for the new network

history_net_layer = net_layer.train(x_tr, Y_tr, epochs=5, batch_size=100, rng=np.random.default_rng(2026))

pred_val_net_layer = np.argmax(net_layer.predict(x_val), axis=1)
print("validation accuracy:", accuracy(y_val, pred_val_net_layer))

validation accuracy: 0.9672
